# Asistente RAG sobre Hades (2018)

Asistente documental que responde preguntas en inglés sobre el videojuego Hades (Supergiant Games) usando solo una base de conocimiento propia, con memoria de conversación.

Flujo: documentos → embeddings → Qdrant → recuperación (denso + BM25) → contexto → LLM → respuesta con fuentes.

- Base documental: hades1.json (wiki de Hades en Fandom, CC BY-SA, limpiada con el scraper) y builds.json (builds escritas por el grupo).
- Embeddings: BAAI/bge-base-en-v1.5.
- Base vectorial: Qdrant en modo local.
- Modelo generativo: Qwen/Qwen2.5-3B-Instruct (Hugging Face).
- Memoria: historial de turnos y reescritura de las preguntas de seguimiento.

Se ejecuta en Google Colab con GPU (Entorno de ejecución → Cambiar tipo de entorno → T4).

## 1. Preparar el entorno

Se fija la misma versión de Transformers que en el notebook de la práctica. Si es la primera instalación, reinicia la sesión (Entorno de ejecución → Reiniciar sesión) y continúa desde la celda de configuración.

In [ ]:
%pip install -q "transformers==4.57.6" sentence-transformers qdrant-client rank-bm25 accelerate pandas

In [ ]:
import os, re, json
os.environ["USE_TF"] = "0"

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from qdrant_client import QdrantClient, models
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 200)

CONFIG = {
    "ruta_hades": "hades1.json",
    "ruta_builds": "builds.json",            # opcional: builds escritas por el grupo
    "modelo_embeddings": "BAAI/bge-base-en-v1.5",
    "prefijo_consulta": "Represent this sentence for searching relevant passages: ",
    "modelo_llm": "Qwen/Qwen2.5-3B-Instruct",
    "coleccion": "hades1",
    "ruta_qdrant": "qdrant_hades",
    "k": 5,
    "turnos_memoria": 4,
}
NO_INFO = "I can't find that information in the available documents."

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Subir los archivos

Sube hades1.json (y builds.json si ya lo tenéis) al entorno de Colab.

In [ ]:
if not os.path.exists(CONFIG["ruta_hades"]):
    from google.colab import files
    print("Sube hades1.json (y, si existe, builds.json)")
    files.upload()

print("hades1.json:", os.path.exists(CONFIG["ruta_hades"]))
print("builds.json:", os.path.exists(CONFIG["ruta_builds"]))

## 3. Base documental

Cada fragmento tiene entidad, sección, categoría, tipo (juego, historia o trivia), fuente, url y texto. El texto empieza por el nombre de la entidad para que se entienda por sí solo cuando se recupera aislado.

Al cargar, se eliminan las frases que mencionan Hades II o Melinoë (la wiki mezcla los dos juegos), los fragmentos demasiado cortos y los duplicados. Las builds del grupo se añaden desde builds.json; las entradas de la plantilla sin rellenar (texto que empieza por corchete) se ignoran.

In [ ]:
PATRON_HADES2 = re.compile(r"Melino[eë]|Hades II", re.IGNORECASE)

def quitar_hades2(doc):
    original = doc["texto"].strip()
    lineas = []
    for linea in original.split("\n"):
        frases = re.split(r"(?<=[.!?])\s+", linea)
        frases = [f for f in frases if not PATRON_HADES2.search(f)]
        if frases:
            lineas.append(" ".join(frases))
    limpio = "\n".join(lineas).strip()
    # si se perdió la cabecera "Entidad - Sección:", se restituye
    if limpio and limpio != original and not limpio.startswith(doc["entidad"]):
        limpio = f"{doc['entidad']} - {doc.get('seccion', '')}: {limpio}"
    return limpio

def cargar_documentos():
    with open(CONFIG["ruta_hades"], encoding="utf-8") as f:
        crudos = json.load(f)

    if os.path.exists(CONFIG["ruta_builds"]):
        with open(CONFIG["ruta_builds"], encoding="utf-8") as f:
            for b in json.load(f):
                if str(b.get("texto", "")).strip().startswith("["):
                    continue
                b.setdefault("entidad", "Build")
                b.setdefault("categoria", "build")
                b.setdefault("tipo", "juego")
                b.setdefault("seccion", "Build")
                b.setdefault("fuente", "grupo (propio)")
                b.setdefault("url", "")
                crudos.append(b)

    docs, vistos, descartados = [], set(), 0
    for d in crudos:
        texto = quitar_hades2(d)
        if len(texto.split()) < 12 or texto in vistos:
            descartados += 1
            continue
        vistos.add(texto)
        docs.append({
            "id": len(docs),
            "entidad": d["entidad"],
            "seccion": d.get("seccion", ""),
            "categoria": d.get("categoria", "general"),
            "tipo": d.get("tipo", "juego"),
            "fuente": d.get("fuente", ""),
            "url": d.get("url", ""),
            "texto": texto,
        })
    print(f"Fragmentos cargados: {len(docs)} (descartados: {descartados})")
    return docs

DOCS = cargar_documentos()

In [ ]:
df_docs = pd.DataFrame(DOCS)
palabras = df_docs["texto"].str.split().str.len()

display(df_docs.groupby("categoria")["tipo"].value_counts().unstack(fill_value=0))
print("Entidades distintas:", df_docs["entidad"].nunique())
print("Palabras por fragmento (mediana / máximo):", palabras.median(), "/", palabras.max())
print("Fragmentos que aún mencionan Hades II o Melinoë:", int(df_docs["texto"].str.contains(PATRON_HADES2).sum()))

## 4. Embeddings

Cada fragmento se convierte en un vector con bge-base-en-v1.5 (768 dimensiones). Los vectores se normalizan, de modo que el producto escalar equivale a la similitud coseno. A las preguntas se les antepone el prefijo de instrucción que recomienda el modelo; a los fragmentos no.

In [ ]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo_embeddings = SentenceTransformer(CONFIG["modelo_embeddings"], device=dispositivo)

EMB = modelo_embeddings.encode(
    [d["texto"] for d in DOCS],
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)
print("Matriz de embeddings:", EMB.shape)

def embedding_pregunta(pregunta):
    return modelo_embeddings.encode(
        CONFIG["prefijo_consulta"] + pregunta,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

## 5. Qdrant

Los vectores se guardan en una colección de Qdrant en modo local (una carpeta, sin servidor). El payload de cada punto contiene todos los campos del fragmento, así que se puede filtrar por tipo o categoría.

Si se reejecuta la celda, se cierra el cliente anterior para evitar el bloqueo del modo local y se recrea la colección. En Colab el disco se borra al terminar la sesión; el índice se reconstruye en segundos a partir de los JSON.

In [ ]:
try:
    cliente_qdrant.close()
except NameError:
    pass

cliente_qdrant = QdrantClient(path=CONFIG["ruta_qdrant"])

if cliente_qdrant.collection_exists(CONFIG["coleccion"]):
    cliente_qdrant.delete_collection(CONFIG["coleccion"])

cliente_qdrant.create_collection(
    collection_name=CONFIG["coleccion"],
    vectors_config=models.VectorParams(size=EMB.shape[1], distance=models.Distance.COSINE),
)

for inicio in range(0, len(DOCS), 256):
    lote = range(inicio, min(inicio + 256, len(DOCS)))
    cliente_qdrant.upsert(
        collection_name=CONFIG["coleccion"],
        points=[models.PointStruct(id=i, vector=EMB[i].tolist(), payload=DOCS[i]) for i in lote],
    )

print("Puntos en Qdrant:", cliente_qdrant.count(collection_name=CONFIG["coleccion"]).count)

## 6. Recuperación

Hay tres piezas:

- Búsqueda densa en Qdrant: encuentra fragmentos con significado parecido a la pregunta.
- Búsqueda BM25: encuentra coincidencias exactas de palabras, útil con nombres propios del juego como Heart-Seeking Bow o Titan Blood.
- Fusión RRF: combina los dos rankings por posición, sin tener que calibrar sus puntuaciones.

Todas las búsquedas filtran por tipo. Por defecto solo se usan fragmentos de tipo juego, para que la historia y la trivia no compitan con los datos de juego.

In [ ]:
TODOS_LOS_TIPOS = ("juego", "historia", "trivia")

def tokenizar(texto):
    return re.findall(r"[a-z0-9']+", texto.lower())

BM25 = BM25Okapi([tokenizar(d["texto"]) for d in DOCS])
TIPOS_DOC = np.array([d["tipo"] for d in DOCS])

def buscar_denso(vector, n, tipos):
    filtro = models.Filter(must=[
        models.FieldCondition(key="tipo", match=models.MatchAny(any=list(tipos)))
    ])
    respuesta = cliente_qdrant.query_points(
        collection_name=CONFIG["coleccion"],
        query=vector.tolist(),
        limit=n,
        query_filter=filtro,
        with_payload=False,
    )
    return [p.id for p in respuesta.points]

def buscar_bm25(pregunta, n, tipos):
    puntos = BM25.get_scores(tokenizar(pregunta))
    puntos = np.where(np.isin(TIPOS_DOC, list(tipos)), puntos, -1.0)
    orden = np.argsort(puntos)[::-1][:n]
    return [int(i) for i in orden if puntos[i] > 0]

def fusion_rrf(listas, c=60):
    acumulado = {}
    for lista in listas:
        for posicion, idx in enumerate(lista):
            acumulado[idx] = acumulado.get(idx, 0.0) + 1.0 / (c + posicion + 1)
    return sorted(acumulado, key=acumulado.get, reverse=True)

def recuperar(pregunta, k=None, modo="hibrido", tipos=("juego",)):
    k = k or CONFIG["k"]
    vector = embedding_pregunta(pregunta)
    if modo == "denso":
        ids = buscar_denso(vector, k, tipos)
    else:
        candidatos = max(30, 4 * k)
        ids = fusion_rrf([
            buscar_denso(vector, candidatos, tipos),
            buscar_bm25(pregunta, candidatos, tipos),
        ])[:k]
    return [{"documento": DOCS[i], "puntuacion": float(EMB[i] @ vector)} for i in ids]

def mostrar_recuperacion(pregunta, **kwargs):
    for pos, r in enumerate(recuperar(pregunta, **kwargs), start=1):
        d = r["documento"]
        print(f"{pos}. [{d['id']}] {d['entidad']} - {d['seccion']} | similitud {r['puntuacion']:.3f}")
        print("   ", d["texto"][:170].replace("\n", " "))

mostrar_recuperacion("Which aspects does the Heart-Seeking Bow have?")

## 7. Modelo generativo y prompt

Qwen2.5-3B-Instruct es un modelo de chat, así que se usa su plantilla de mensajes (sistema, usuario, asistente). El prompt de sistema le obliga a responder solo con los fragmentos numerados, a citar sus ids y a usar una frase fija cuando no hay información, lo que permite detectar la abstención automáticamente.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["modelo_llm"])
llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["modelo_llm"], torch_dtype=torch.float16, device_map="auto"
)
llm.eval()

def generar(mensajes, max_new_tokens=300):
    texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(texto, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        salida = llm.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
    nuevos = salida[0][entradas["input_ids"].shape[1]:]
    return tokenizer.decode(nuevos, skip_special_tokens=True).strip()

SYSTEM_RAG = (
    "You are an expert assistant for the video game Hades (2018) by Supergiant Games. "
    "Answer the question using ONLY the numbered CONTEXT passages. "
    f"If the context does not contain the answer, reply exactly: {NO_INFO} "
    "Do not use outside knowledge and do not talk about Hades II. "
    "Be concise (at most 5 sentences) and cite the ids of the passages you used, like [12] [40]."
)

def construir_contexto(resultados):
    return "\n\n".join(f"[{r['documento']['id']}] {r['documento']['texto']}" for r in resultados)

## 8. Asistente con memoria

La clase AsistenteHades une recuperación y generación y mantiene el historial de la conversación.

La memoria funciona en dos niveles:

- Reescritura de la pregunta: una pregunta como "And what does the one named after Hera do?" no sirve para buscar. Antes de recuperar, el modelo la convierte en una pregunta autónoma usando el historial, y la búsqueda se hace con esa.
- Historial en el prompt: los últimos turnos (pregunta y respuesta) se pasan como mensajes del chat para que la respuesta mantenga el hilo.

Parámetros para comparar variantes: modo (denso o híbrido), tipos, k, umbral de similitud mínima y memoria. Si la mejor similitud no llega al umbral, el asistente se abstiene sin llamar al modelo.

In [ ]:
SYSTEM_CONDENSAR = (
    "Rewrite the user's last question as a standalone question, using the conversation to resolve "
    "pronouns and references such as 'it', 'that one' or 'its'. Keep game names exactly as written. "
    "Output only the rewritten question and nothing else."
)

class AsistenteHades:
    def __init__(self, modo="hibrido", tipos=("juego",), k=None, umbral=None, memoria=True, turnos=None):
        self.modo = modo
        self.tipos = tipos
        self.k = k or CONFIG["k"]
        self.umbral = umbral
        self.memoria = memoria
        self.turnos = turnos or CONFIG["turnos_memoria"]
        self.historial = []

    def reiniciar(self):
        self.historial = []

    def _autonoma(self, pregunta):
        if not (self.memoria and self.historial):
            return pregunta
        conversacion = "\n".join(f"User: {p}\nAssistant: {r}" for p, r in self.historial[-self.turnos:])
        mensajes = [
            {"role": "system", "content": SYSTEM_CONDENSAR},
            {"role": "user", "content": f"Conversation:\n{conversacion}\n\nLast question: {pregunta}"},
        ]
        nueva = generar(mensajes, max_new_tokens=60).split("\n")[0].strip()
        return nueva if 5 <= len(nueva) <= 300 else pregunta

    def preguntar(self, pregunta):
        autonoma = self._autonoma(pregunta)
        resultados = recuperar(autonoma, k=self.k, modo=self.modo, tipos=self.tipos)
        mejor = max((r["puntuacion"] for r in resultados), default=0.0)

        if not resultados or (self.umbral is not None and mejor < self.umbral):
            respuesta = NO_INFO
        else:
            mensajes = [{"role": "system", "content": SYSTEM_RAG}]
            if self.memoria:
                for p, r in self.historial[-self.turnos:]:
                    mensajes += [{"role": "user", "content": p}, {"role": "assistant", "content": r}]
            mensajes.append({
                "role": "user",
                "content": f"CONTEXT:\n{construir_contexto(resultados)}\n\nQUESTION: {autonoma}",
            })
            respuesta = generar(mensajes)

        if self.memoria:
            self.historial.append((pregunta, respuesta))

        return {
            "pregunta": pregunta,
            "pregunta_autonoma": autonoma,
            "respuesta": respuesta,
            "recuperados": resultados,
            "mejor_similitud": mejor,
        }

def mostrar(r):
    print("QUESTION:", r["pregunta"])
    if r["pregunta_autonoma"] != r["pregunta"]:
        print("STANDALONE QUESTION:", r["pregunta_autonoma"])
    print("ANSWER:", r["respuesta"])
    print("SOURCES:")
    for x in r["recuperados"]:
        d = x["documento"]
        print(f"  [{d['id']}] {d['entidad']} - {d['seccion']} ({x['puntuacion']:.3f}) {d['url']}")

## 9. Pruebas

Se definen 12 preguntas con respuesta en la colección y 3 sin respuesta. Cada pregunta con respuesta lleva palabras clave que deben aparecer en el fragmento que la contiene; sirven para medir la recuperación de forma automática. Las preguntas sin respuesta incluyen una sobre Hades II, otra sobre ventas y otra sobre requisitos técnicos.

In [ ]:
PRUEBAS_DENTRO = [
    ("Which aspects does the Stygian Blade have?", ["Nemesis", "Poseidon", "Arthur"]),
    ("What does the Aspect of Arthur give you?", ["Excalibur", "+50"]),
    ("What does the Aspect of Nemesis do for the Stygian Blade?", ["Critical", "Special"]),
    ("Which aspects does the Heart-Seeking Bow have?", ["Chiron", "Hera", "Rama"]),
    ("What does the Aspect of Hera do for the Heart-Seeking Bow?", ["Cast loads into your next Attack"]),
    ("What does the Hard Labor condition do in the Pact of Punishment?", ["Hard Labor", "+20%"]),
    ("How many encounters does a keepsake need to reach rank 2?", ["25 encounters"]),
    ("How many tasks are there in the Fated List of Minor Prophecies?", ["55 tasks"]),
    ("What are Duo Boons?", ["combine the powers of two different gods"]),
    ("How much damage does the Bone Hydra's Lunge deal below 66% health?", ["Lunge", "increases to 16"]),
    ("How can I get Darkness?", ["Infernal Troves"]),
    ("What happens when you pick one god's boon in a Trial of the Gods?", ["spurned"]),
]

PRUEBAS_FUERA = [
    "What is Melinoë's main weapon in Hades II?",
    "How many copies of Hades has Supergiant Games sold worldwide?",
    "Which graphics card do I need to play Hades at 4K?",
]

def contiene_claves(doc, claves):
    return all(c.lower() in doc["texto"].lower() for c in claves)

# Comprobación: cada pregunta con respuesta debe tener al menos un fragmento que contenga sus claves
for pregunta, claves in PRUEBAS_DENTRO:
    n = sum(contiene_claves(d, claves) for d in DOCS)
    print(f"{n:3d} fragmentos con las claves | {pregunta}")

In [ ]:
asistente = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=None, memoria=True)

for pregunta, _ in PRUEBAS_DENTRO[:5]:
    asistente.reiniciar()
    print("=" * 90)
    mostrar(asistente.preguntar(pregunta))

In [ ]:
for pregunta in PRUEBAS_FUERA:
    asistente.reiniciar()
    print("=" * 90)
    mostrar(asistente.preguntar(pregunta))

## 10. Evaluación automática de la recuperación

Para cada pregunta con respuesta se comprueba si alguno de los K fragmentos recuperados contiene las palabras clave (hit@k) y en qué posición aparece el primero (MRR). Se comparan tres variantes de recuperación y varios valores de K.

In [ ]:
def evaluar_recuperacion(k=5, modo="hibrido", tipos=("juego",)):
    aciertos, rr = 0, []
    for pregunta, claves in PRUEBAS_DENTRO:
        resultados = recuperar(pregunta, k=k, modo=modo, tipos=tipos)
        posicion = next((i + 1 for i, r in enumerate(resultados) if contiene_claves(r["documento"], claves)), None)
        aciertos += posicion is not None
        rr.append(1 / posicion if posicion else 0.0)
    return {"hit@k": round(aciertos / len(PRUEBAS_DENTRO), 2), "MRR": round(float(np.mean(rr)), 2)}

variantes = [
    ("denso, todos los tipos", "denso", TODOS_LOS_TIPOS),
    ("denso, solo juego", "denso", ("juego",)),
    ("híbrido, solo juego", "hibrido", ("juego",)),
]
tabla_recuperacion = pd.DataFrame(
    [{"variante (k=5)": nombre, **evaluar_recuperacion(5, modo, tipos)} for nombre, modo, tipos in variantes]
)
display(tabla_recuperacion)

tabla_k = pd.DataFrame(
    [{"k": k, **evaluar_recuperacion(k, "hibrido", ("juego",))} for k in [1, 3, 5, 8]]
)
display(tabla_k)

## 11. Umbral de similitud

Recuperar siempre devuelve los fragmentos más parecidos, aunque la respuesta no esté. Se compara la mejor similitud de las preguntas con respuesta con la de las preguntas sin respuesta. Solo se fija un umbral si los dos grupos se separan; con 15 preguntas la calibración es orientativa y debería repetirse con más casos.

In [ ]:
dentro = [recuperar(p, k=1, modo="denso")[0]["puntuacion"] for p, _ in PRUEBAS_DENTRO]
fuera = [recuperar(p, k=1, modo="denso")[0]["puntuacion"] for p in PRUEBAS_FUERA]

print(f"Con respuesta: mínimo {min(dentro):.3f}, media {np.mean(dentro):.3f}")
print(f"Sin respuesta: máximo {max(fuera):.3f}, media {np.mean(fuera):.3f}")

if max(fuera) < min(dentro):
    UMBRAL = round((max(fuera) + min(dentro)) / 2, 3)
    print("Los grupos se separan. Umbral propuesto:", UMBRAL)
else:
    UMBRAL = None
    print("Los grupos se solapan: la similitud sola no permite abstenerse de forma fiable.")
    print("Se confía en la instrucción de abstención del prompt.")

## 12. Evaluación manual

La tabla recoge la respuesta y las fuentes de cada pregunta, una comprobación aproximada de las palabras clave y las columnas para puntuar de 1 a 5 cada criterio (recuperación, fidelidad, corrección, relevancia, trazabilidad y abstención) con observaciones. La comprobación automática es solo una ayuda: la corrección y la fidelidad se juzgan leyendo la respuesta y los fragmentos.

In [ ]:
def abstiene(respuesta):
    return "can't find" in respuesta.replace("’", "'").lower()

asistente = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=UMBRAL, memoria=True)
filas = []

for pregunta, claves in PRUEBAS_DENTRO:
    asistente.reiniciar()
    r = asistente.preguntar(pregunta)
    filas.append({
        "pregunta": pregunta,
        "esperada": "con respuesta",
        "respuesta": r["respuesta"],
        "fuentes (ids)": [x["documento"]["id"] for x in r["recuperados"]],
        "claves en la respuesta": f"{sum(c.lower() in r['respuesta'].lower() for c in claves)}/{len(claves)}",
        "se abstiene": abstiene(r["respuesta"]),
    })

for pregunta in PRUEBAS_FUERA:
    asistente.reiniciar()
    r = asistente.preguntar(pregunta)
    filas.append({
        "pregunta": pregunta,
        "esperada": "sin respuesta",
        "respuesta": r["respuesta"],
        "fuentes (ids)": [x["documento"]["id"] for x in r["recuperados"]],
        "claves en la respuesta": "-",
        "se abstiene": abstiene(r["respuesta"]),
    })

evaluacion = pd.DataFrame(filas)
for col in ["recuperacion (1-5)", "fidelidad (1-5)", "correccion (1-5)", "relevancia (1-5)", "trazabilidad (1-5)", "abstencion (1-5)"]:
    evaluacion[col] = None
evaluacion["observaciones"] = ""

evaluacion

## 13. Mejora: comparación con el sistema básico

Se compara un sistema básico (búsqueda densa, todos los tipos de fragmento, sin umbral, sin memoria) con el sistema final (búsqueda híbrida, solo fragmentos de juego, umbral si se pudo calibrar y memoria).

Métricas: preguntas cuyo fragmento correcto se recupera, media de palabras clave que aparecen en la respuesta (aproximada) y abstenciones correctas en las preguntas sin respuesta.

In [ ]:
def evaluar_sistema(**config):
    sistema = AsistenteHades(**config)
    hits, cobertura, abstenciones = [], [], []
    for pregunta, claves in PRUEBAS_DENTRO:
        sistema.reiniciar()
        r = sistema.preguntar(pregunta)
        hits.append(any(contiene_claves(x["documento"], claves) for x in r["recuperados"]))
        cobertura.append(np.mean([c.lower() in r["respuesta"].lower() for c in claves]))
    for pregunta in PRUEBAS_FUERA:
        sistema.reiniciar()
        abstenciones.append(abstiene(sistema.preguntar(pregunta)["respuesta"]))
    return {
        "recuperación correcta": f"{sum(hits)}/{len(hits)}",
        "claves en la respuesta (media)": round(float(np.mean(cobertura)), 2),
        "abstenciones correctas": f"{sum(abstenciones)}/{len(abstenciones)}",
    }

comparacion = pd.DataFrame({
    "básico (denso, sin filtro, sin umbral)": evaluar_sistema(modo="denso", tipos=TODOS_LOS_TIPOS, umbral=None, memoria=False),
    "final (híbrido, filtro, umbral)": evaluar_sistema(modo="hibrido", tipos=("juego",), umbral=UMBRAL, memoria=True),
}).T
comparacion

## 14. Mejora de la memoria

Se repite un diálogo de dos turnos con y sin memoria. En el segundo turno la pregunta no nombra ni el arma ni el aspecto, así que sin memoria la búsqueda no puede encontrar el fragmento correcto.

In [ ]:
DIALOGO = [
    ("Which aspects does the Heart-Seeking Bow have?", ["Chiron", "Hera", "Rama"]),
    ("And what does the one named after Hera do?", ["Cast loads into your next Attack"]),
]

for memoria in (False, True):
    sistema = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=None, memoria=memoria)
    print("=" * 90)
    print("MEMORIA:", memoria)
    for pregunta, claves in DIALOGO:
        r = sistema.preguntar(pregunta)
        correcto = any(contiene_claves(x["documento"], claves) for x in r["recuperados"])
        print("\nQ:", pregunta)
        print("pregunta usada para buscar:", r["pregunta_autonoma"])
        print("A:", r["respuesta"])
        print("fragmento correcto recuperado:", correcto)

## 15. Demostración interactiva

Conversación libre con memoria para la demostración en clase. Para terminar, escribe exit o deja la línea vacía. El comando reiniciar borra la memoria.

In [ ]:
demo = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=UMBRAL, memoria=True)

while True:
    entrada = input("You: ").strip()
    if entrada.lower() in {"", "exit", "quit"}:
        break
    if entrada.lower() == "reiniciar":
        demo.reiniciar()
        print("(memoria borrada)")
        continue
    mostrar(demo.preguntar(entrada))
    print()

## 16. Builds del grupo

La wiki no contiene builds, así que las escribe el grupo en builds.json (ver la plantilla builds_plantilla.json). Cada build se divide en 2 o 3 fragmentos: idea general, bendiciones y mejoras prioritarias, y puntos débiles.

Reglas para que se recuperen bien:

- Texto en inglés y con palabras propias.
- Cada fragmento nombra el arma, el aspecto y los dioses implicados, para entenderse por sí solo.
- Una sola idea por fragmento, de 40 a 120 palabras.
- Solo datos que hayáis comprobado en el juego o en la wiki.

Después de añadir o editar builds.json, vuelve a ejecutar desde la sección 3 (carga, embeddings, Qdrant y BM25).

## 17. Descripción, resultados y limitaciones

Temática y documentos: asistente sobre Hades (2018). La base documental procede de la wiki de Hades en Fandom (texto CC BY-SA, con la url de cada fragmento) y de las builds escritas por el grupo. Los fragmentos se organizan por entidad, sección, categoría y tipo.

Respuestas para la presentación:

- Organización de los documentos: fragmentos de una idea, con la entidad al inicio del texto, metadatos y filtro por tipo.
- Modelo de embeddings: BAAI/bge-base-en-v1.5, vectores normalizados y similitud coseno en Qdrant.
- Selección de documentos: búsqueda híbrida (densa y BM25) fusionada con RRF, filtro por tipo y K=5.
- Modelo generativo: Qwen/Qwen2.5-3B-Instruct, con plantilla de chat y generación determinista.
- Comprobación de fidelidad: prompt restrictivo, citas con el id del fragmento, fuentes mostradas y evaluación manual.
- Preguntas fuera de la colección: frase fija de abstención y umbral de similitud si la calibración lo permite.
- Siguiente versión: reranker, más preguntas de evaluación, más builds y revisión de los datos frente al juego.

Limitaciones: la wiki puede tener errores o datos cambiados con parches; el conjunto de pruebas es pequeño y la calibración del umbral es orientativa; un modelo de 3.000 millones de parámetros puede mezclar datos de fragmentos parecidos; las builds dependen de lo que escriba el grupo; las comprobaciones automáticas de palabras clave son aproximadas.

Rellena con vuestros resultados reales las tablas de las secciones 10, 12, 13 y 14 y comenta si cada mejora ha funcionado.

## 18. Conclusiones

Flujo implementado: documentos → embeddings → Qdrant → recuperación híbrida → contexto → LLM → respuesta con fuentes y memoria.

La calidad depende de la base documental, de cómo se fragmenta, del modelo de embeddings, de la recuperación y del modelo generativo. RAG reduce las invenciones pero no las elimina: hay que revisar siempre las respuestas y sus fuentes.